In [ ]:
import pypsa
import pandas as pd
import numpy as np

# ============================
# INPUTS / OUTPUTS
# ============================
FINE_PATH   = "base_s_512_elec.nc"
COARSE_PATH = "elec_s_39_optimized.nc"
OUT_PATH    = "base_s_512_elec_seeded_from_39_spain.nc"

COUNTRY = "ES"

# We only want to seed these (hydro/PHS stay untouched / fixed)
RES_GEN_CARRIERS = ["solar", "solar-hsat", "onwind", "offwind-ac", "offwind-dc", "offwind-float"]

# Battery in your 512-bus model is NOT a storage_unit. It's stores + links:
BATTERY_STORE_CARRIER = "battery"
BATTERY_LINK_CARRIERS = ["battery charger", "battery discharger"]

# If your coarse (39-node) gives only one "solar" carrier, decide what to do:
# Option A: seed only "solar" (leave solar-hsat unchanged)  -> set SOLAR_MODE="solar_only"
# Option B: split coarse "solar" across ("solar"+"solar-hsat") in fine -> set SOLAR_MODE="split_solar"
SOLAR_MODE = "split_solar"  # recommended if coarse has only "solar"

# ============================
# HELPERS
# ============================
def _country_buses(n: pypsa.Network, country: str) -> pd.Index:
    if "country" not in n.buses.columns:
        raise ValueError("n.buses has no 'country' column.")
    buses = n.buses.index[n.buses["country"] == country]
    if len(buses) == 0:
        raise ValueError(f"No buses for country='{country}'.")
    return buses

def _cap_col(df: pd.DataFrame) -> str:
    return "p_nom_opt" if "p_nom_opt" in df.columns else "p_nom"

def _cap_series(df: pd.DataFrame, col: str) -> pd.Series:
    """Missing/NaN/inf/<=0 treated as +inf."""
    if col in df.columns:
        s = df[col].copy()
        bad = ~np.isfinite(s) | (s <= 0)
        s.loc[bad] = np.inf
        return s.astype(float)
    return pd.Series(np.inf, index=df.index, dtype=float)

def _waterfill(ids: pd.Index, headroom: pd.Series, weights: pd.Series, total: float) -> pd.Series:
    """Weighted water-filling allocation respecting headroom."""
    alloc = pd.Series(0.0, index=ids, dtype=float)
    live = ids.copy()
    remaining = float(total)

    hr = headroom.reindex(ids).clip(lower=0).copy()
    w  = weights.reindex(ids).clip(lower=0).copy()
    if w.loc[live].sum() <= 0:
        w.loc[live] = 1.0

    while remaining > 1e-9 and len(live) > 0:
        wl = w.loc[live]
        wl = wl / wl.sum()
        proposal = wl * remaining
        take = np.minimum(proposal, hr.loc[live])

        alloc.loc[live] += take
        remaining -= float(take.sum())

        hr.loc[live] -= take
        live = hr[hr > 1e-9].index

    return alloc

def targets_from_coarse_39(n39: pypsa.Network, country: str) -> tuple[dict, dict]:
    """
    Sum targets across ALL buses of this country in the 39-node model.
    Returns:
      gen_targets_mw: carrier -> MW (from generators p_nom_opt)
      batt_power_mw:  MW (from storage_units carrier 'battery' p_nom_opt if present)
    """
    buses = _country_buses(n39, country)
    keep = set(buses)

    # generators
    g = n39.generators[n39.generators.bus.isin(keep)].copy()
    if "p_nom_opt" not in g.columns:
        raise ValueError("Coarse network has no generators.p_nom_opt (not optimized?)")
    gen_targets = g.groupby("carrier")["p_nom_opt"].sum().to_dict()

    # battery power target (if the coarse model stores it as storage_units carrier 'battery')
    batt_power = 0.0
    if hasattr(n39, "storage_units") and not n39.storage_units.empty and "p_nom_opt" in n39.storage_units.columns:
        su = n39.storage_units[n39.storage_units.bus.isin(keep)].copy()
        if not su.empty:
            by_car = su.groupby("carrier")["p_nom_opt"].sum().to_dict()
            batt_power = float(by_car.get("battery", 0.0))

    # keep only the RES carriers we care about (others ignored)
    gen_targets_mw = {c: float(gen_targets.get(c, 0.0)) for c in RES_GEN_CARRIERS}

    # special handling: coarse may have only "solar" while fine has solar + solar-hsat
    # if split_solar: add coarse solar into both and then later allocate across both together
    # (we implement allocation across a "solar family" below)
    return gen_targets_mw, batt_power

def seed_spain_res_generators(n_fine: pypsa.Network, country: str, gen_targets_mw: dict):
    """
    Overwrite p_nom for Spain RES generator carriers to match targets (MW).
    Hydro/PHS not touched because they are storage_units, not generators.
    """
    buses = _country_buses(n_fine, country)
    keep = set(buses)

    # ---- SOLAR handling ----
    if SOLAR_MODE == "split_solar":
        solar_target = float(gen_targets_mw.get("solar", 0.0))  # coarse solar
        # allocate across BOTH fine carriers present: solar + solar-hsat
        solar_family = ["solar", "solar-hsat"]
        idx = n_fine.generators.index[
            n_fine.generators.bus.isin(keep) & n_fine.generators.carrier.isin(solar_family)
        ]
        if len(idx) > 0 and solar_target > 0:
            try:
                w = n_fine.generators_t.p_max_pu[idx].mean().reindex(idx).fillna(0.0)
                if w.sum() <= 0:
                    w[:] = 1.0
            except Exception:
                w = pd.Series(1.0, index=idx)

            # overwrite only these solar-family gens
            n_fine.generators.loc[idx, "p_nom"] = 0.0
            headroom = _cap_series(n_fine.generators.loc[idx], "p_nom_max")
            alloc = _waterfill(idx, headroom=headroom, weights=w, total=solar_target)
            n_fine.generators.loc[idx, "p_nom"] = alloc
            n_fine.generators.loc[idx, "p_nom_extendable"] = False
            print(f"[seed RES] solar-family (solar+solar-hsat): target={solar_target:.2f} MW, allocated={alloc.sum():.2f} MW")
        else:
            print("[seed RES] solar-family: no assets or target=0, skipped.")
    else:
        # seed only 'solar' carrier, leave solar-hsat unchanged
        carrier = "solar"
        target = float(gen_targets_mw.get(carrier, 0.0))
        idx = n_fine.generators.index[
            n_fine.generators.bus.isin(keep) & (n_fine.generators.carrier == carrier)
        ]
        if len(idx) > 0:
            try:
                w = n_fine.generators_t.p_max_pu[idx].mean().reindex(idx).fillna(0.0)
                if w.sum() <= 0:
                    w[:] = 1.0
            except Exception:
                w = pd.Series(1.0, index=idx)

            n_fine.generators.loc[idx, "p_nom"] = 0.0
            headroom = _cap_series(n_fine.generators.loc[idx], "p_nom_max")
            alloc = _waterfill(idx, headroom=headroom, weights=w, total=target)
            n_fine.generators.loc[idx, "p_nom"] = alloc
            n_fine.generators.loc[idx, "p_nom_extendable"] = False
            print(f"[seed RES] solar: target={target:.2f} MW, allocated={alloc.sum():.2f} MW")
        else:
            print("[seed RES] solar: no assets, skipped.")

    # ---- Other RES carriers (exclude solar-hsat because we handled it above) ----
    for carrier, target in gen_targets_mw.items():
        if carrier in ["solar-hsat"] or (carrier == "solar" and SOLAR_MODE == "split_solar"):
            continue
        if carrier not in RES_GEN_CARRIERS:
            continue

        idx = n_fine.generators.index[
            n_fine.generators.bus.isin(keep) & (n_fine.generators.carrier == carrier)
        ]
        if len(idx) == 0:
            print(f"[seed RES] {carrier}: no assets, skipped.")
            continue

        try:
            w = n_fine.generators_t.p_max_pu[idx].mean().reindex(idx).fillna(0.0)
            if w.sum() <= 0:
                w[:] = 1.0
        except Exception:
            w = pd.Series(1.0, index=idx)

        n_fine.generators.loc[idx, "p_nom"] = 0.0
        headroom = _cap_series(n_fine.generators.loc[idx], "p_nom_max")
        alloc = _waterfill(idx, headroom=headroom, weights=w, total=float(target))
        n_fine.generators.loc[idx, "p_nom"] = alloc
        n_fine.generators.loc[idx, "p_nom_extendable"] = False
        print(f"[seed RES] {carrier}: target={target:.2f} MW, allocated={alloc.sum():.2f} MW")

def seed_spain_battery_power_in_links(n_fine: pypsa.Network, country: str, battery_power_mw: float):
    """
    Seed battery POWER capacity (MW) in Spain.
    In this dataset, battery power is in links: 'battery charger' and 'battery discharger'.
    We set BOTH sums to battery_power_mw.
    """
    if battery_power_mw <= 0:
        print("[seed battery] target=0 MW, skipped.")
        return
    if n_fine.links.empty:
        print("[seed battery] no links table, skipped.")
        return

    buses = _country_buses(n_fine, country)
    keep = set(buses)

    for link_carrier in BATTERY_LINK_CARRIERS:
        idx = n_fine.links.index[
            (n_fine.links.carrier == link_carrier) &
            (n_fine.links.bus0.isin(keep)) &
            (n_fine.links.bus1.isin(keep))
        ]
        if len(idx) == 0:
            print(f"[seed battery] no Spain links for '{link_carrier}', skipped.")
            continue

        # simple equal weights (you can replace by bus-load weights if you want)
        w = pd.Series(1.0, index=idx, dtype=float)

        n_fine.links.loc[idx, "p_nom"] = 0.0
        headroom = _cap_series(n_fine.links.loc[idx], "p_nom_max")
        alloc = _waterfill(idx, headroom=headroom, weights=w, total=float(battery_power_mw))

        n_fine.links.loc[idx, "p_nom"] = alloc
        n_fine.links.loc[idx, "p_nom_extendable"] = False
        print(f"[seed battery] {link_carrier}: target={battery_power_mw:.2f} MW, allocated={alloc.sum():.2f} MW")


# ============================
# RUN (STEP 1)
# ============================
print("Loading fine:", FINE_PATH)
n_fine = pypsa.Network(FINE_PATH)

print("Loading coarse:", COARSE_PATH)
n39 = pypsa.Network(COARSE_PATH)

gen_targets_mw, battery_power_mw = targets_from_coarse_39(n39, COUNTRY)

print("\nSpain targets from coarse (MW):")
print("  RES generators:", gen_targets_mw)
print("  battery power :", battery_power_mw)

# Seed ONLY renewables + battery power. Hydro/PHS untouched.
seed_spain_res_generators(n_fine, COUNTRY, gen_targets_mw)
seed_spain_battery_power_in_links(n_fine, COUNTRY, battery_power_mw)

n_fine.export_to_netcdf(OUT_PATH)
print("\nSaved:", OUT_PATH)